In [ ]:
import os
import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# Local modules
from model_anchor import DistanceGatedGeoVDSR, DistanceGatedTopoLoss
from curriculum_dataset import CurriculumHMATensorDataset, create_curriculum_dataloaders
from hann_merge import HannStreamMerger

torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
TRAIN_DIRS = [
    r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\train",
    r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\train",
]

VAL_DIRS = [
    r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\validation_contiguous",
    r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\validation_contiguous",
]

CHECKPOINT_DIR = Path(r"D:\Vaibhav\vdsr-atl08\checkpoints_dg_vdsr")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "dg_vdsr_lidar_curriculum"

# -----------------------------------------------------------------------------
# Training hyperparameters
# -----------------------------------------------------------------------------
EPOCHS = 500
BEST_METRIC_NAME = "val_slope_rmse"

# NEW: Set this to a path to resume training, or leave as None to start fresh
RESUME_CHECKPOINT = None 
# Example: RESUME_CHECKPOINT = r"D:\Vaibhav\vdsr-atl08\checkpoints_dg_vdsr\dg_vdsr_lidar_curriculum_epoch_045.pt"

BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0

TOPO_LAYERS = 18
FEATURES = 64

# Loss weights
LOSS_ALPHA = 1.0
LOSS_BETA = 8.0
LOSS_GAMMA = 30.0
LOSS_LAMBDA_PIN = 3.0
PIN_BETA = 2.0
DECAY_RADIUS = 20.0
BUFFER_SIZE = 5

# Dataloader settings
TRAIN_CROP = 128
VAL_CROP = 256
VAL_OVERLAP = 192
NUM_WORKERS = 8  # Now using 8 to leverage the Xeon processor!

# Validation tiling mini-batch size
VAL_PATCH_BATCH = 64  # Increased to feed the P4000 efficiently

# Training control
EARLY_STOPPING_PATIENCE = 500

BEST_METRIC_NAME = "val_slope_rmse"  # scheduler / checkpoint selection metric

training_config = {
    'run_name': RUN_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'topo_layers': TOPO_LAYERS,
    'features': FEATURES,
    'loss_alpha': LOSS_ALPHA,
    'loss_beta': LOSS_BETA,
    'loss_gamma': LOSS_GAMMA,
    'loss_lambda_pin': LOSS_LAMBDA_PIN,
    'pin_beta': PIN_BETA,
    'decay_radius': DECAY_RADIUS,
    'buffer_size': BUFFER_SIZE,
    'train_crop': TRAIN_CROP,
    'val_crop': VAL_CROP,
    'val_overlap': VAL_OVERLAP,
    'val_patch_batch': VAL_PATCH_BATCH,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'best_metric_name': BEST_METRIC_NAME,
    'train_dirs': TRAIN_DIRS,
    'val_dirs': VAL_DIRS,
}

print(json.dumps(training_config, indent=2))


# -----------------------------------------------------------------------------
# Train / validation helpers
# -----------------------------------------------------------------------------
def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

def compute_psnr(pred, gt, eps=1e-8):
    mse = torch.mean((pred - gt) ** 2)
    data_range = torch.clamp(gt.max() - gt.min(), min=eps)
    if mse <= eps:
        return torch.tensor(float('inf'), device=pred.device)
    return 20.0 * torch.log10(data_range) - 10.0 * torch.log10(torch.clamp(mse, min=eps))

def safe_conv(tensor, kernel):
    padded = F.pad(tensor, (1, 1, 1, 1), mode='replicate')
    return F.conv2d(padded, kernel)

sobel_x = torch.tensor(
    [[-1, 0, 1],
     [-2, 0, 2],
     [-1, 0, 1]],
    dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

sobel_y = torch.tensor(
    [[-1, -2, -1],
     [0, 0, 0],
     [1, 2, 1]],
    dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

laplacian = torch.tensor(
    [[0, 1, 0],
     [1, -4, 1],
     [0, 1, 0]],
    dtype=torch.float32
).view(1, 1, 3, 3)

def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    best_metric,
    best_epoch,
    train_history,
    checkpoint_dir,
    training_config,
    is_best=False,
):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'best_metric_name': training_config.get('best_metric_name', 'val_slope_rmse'),
        'train_history': train_history,
        'training_config': training_config,
        'torch_version': torch.__version__,
        'optimizer_hyperparams': {
            'lr': optimizer.param_groups[0].get('lr', None),
            'betas': optimizer.param_groups[0].get('betas', None),
            'eps': optimizer.param_groups[0].get('eps', None),
            'weight_decay': optimizer.param_groups[0].get('weight_decay', None),
        },
    }

    filename = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}.pt"
    torch.save(ckpt, filename)

    meta_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}_meta.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({
            'epoch': epoch,
            'best_metric': best_metric,
            'best_epoch': best_epoch,
            'best_metric_name': training_config.get('best_metric_name', 'val_slope_rmse'),
            'training_config': training_config,
            'optimizer_hyperparams': ckpt['optimizer_hyperparams'],
            'torch_version': torch.__version__,
        }, f, indent=2)

    if is_best:
        best_path = checkpoint_dir / f"{RUN_NAME}_best.pt"
        torch.save(ckpt, best_path)

    return filename


def train_one_epoch(model, criterion, optimizer, train_loader, device, grad_clip_norm=1.0):
    model.train()

    running = {
        'total': 0.0, 'base': 0.0, 'slope': 0.0, 
        'curve': 0.0, 'pin': 0.0, 'mae': 0.0, 'rmse': 0.0, 'anchor_mae': 0.0,
    }
    n_batches = 0

    pbar = tqdm(train_loader, desc='Train', leave=False, dynamic_ncols=True)

    for batch in pbar:
        # 1. Move data to GPU
        dem_bic = batch['dem_bic'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_delta = batch['lidar_delta'].to(device, non_blocking=True, dtype=torch.float32)
        mask = batch['mask'].to(device, non_blocking=True, dtype=torch.float32)
        dist_map = batch['dist_map'].to(device, non_blocking=True, dtype=torch.float32)
        gt_dem = batch['gt_dem'].to(device, non_blocking=True, dtype=torch.float32)

        lidar_raw = dem_bic + lidar_delta

        optimizer.zero_grad(set_to_none=True)

        # 2. Native FP32 Forward pass
        pred_dem, alpha, r_topo, r_anchor = model(dem_bic, lidar_delta, mask, dist_map)
        loss_dict = criterion(pred_dem, gt_dem, lidar_raw, mask, dist_map)

        # 3. Native FP32 Backward pass
        loss_dict['total'].backward()

        if grad_clip_norm is not None and grad_clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

        # 4. Step optimizer
        optimizer.step()

        with torch.no_grad():
            batch_mae = F.l1_loss(pred_dem, gt_dem)
            batch_rmse = compute_rmse(pred_dem, gt_dem)

            # Calculate error strictly at the LiDAR locations
            mask_sum = mask.sum()
            if mask_sum > 0:
                batch_anchor_mae = (mask * torch.abs(pred_dem - lidar_raw)).sum() / mask_sum
            else:
                batch_anchor_mae = torch.tensor(0.0, device=device)

        running['total'] += float(loss_dict['total'].item())
        running['base'] += float(loss_dict['base'].item())
        running['slope'] += float(loss_dict['slope'].item())
        running['curve'] += float(loss_dict['curve'].item())
        running['pin'] += float(loss_dict['pin'].item())
        running['mae'] += float(batch_mae.item())
        running['rmse'] += float(batch_rmse.item())
        running['anchor_mae'] += float(batch_anchor_mae.item())
        n_batches += 1

        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.4f}",
            'mae': f"{batch_mae.item():.4f}",
            'rmse': f"{batch_rmse.item():.4f}",
            'lr': f"{optimizer.param_groups[0]['lr']:.2e}",
        })

    for k in running:
        running[k] /= max(n_batches, 1)

    return running

@torch.inference_mode()
def validate_one_epoch(model, criterion, val_loader, device, val_patch_batch=64):
    model.eval()

    sobel_x_device = sobel_x.to(device)
    sobel_y_device = sobel_y.to(device)
    laplacian_device = laplacian.to(device)

    total_loss = 0.0
    total_mae = 0.0
    total_rmse = 0.0
    total_psnr = 0.0
    total_slope_rmse = 0.0
    total_curve_rmse = 0.0
    total_anchor_mae = 0.0
    n_samples = 0

    outer = tqdm(val_loader, desc='Validate files', leave=False, dynamic_ncols=True)

    for sample_idx, batch in enumerate(outer, start=1):
        dem_bic_all = batch['dem_bic'].squeeze(0).to(device, dtype=torch.float32)
        lidar_delta_all = batch['lidar_delta'].squeeze(0).to(device, dtype=torch.float32)
        mask_all = batch['mask'].squeeze(0).to(device, dtype=torch.float32)
        dist_map_all = batch['dist_map'].squeeze(0).to(device, dtype=torch.float32)
        gt_dem_all = batch['gt_dem'].squeeze(0).to(device, dtype=torch.float32)

        patch_mean_all = batch['patch_mean'].squeeze(0)
        coords_all = batch['coords'].squeeze(0)
        canvas_shape = batch['canvas_shape'].squeeze(0).tolist()
        pad = batch["pad"].item()
        
        original_shape = (
            batch["original_shape"]
            .squeeze(0)
            .tolist()
        )

        gt_canvas = (
            batch['gt_canvas_full']
            .squeeze(0)
            .to(device, dtype=torch.float32)
            .unsqueeze(0)
            .unsqueeze(0)
        )

        merger = HannStreamMerger(
            canvas_shape=canvas_shape,
            patch_size=256,
            device="cuda",
            pad=pad,
            original_shape=original_shape,
        )

        patch_preds = []
        patch_count = dem_bic_all.shape[0]
        n_patch_batches = math.ceil(patch_count / val_patch_batch)

        inner = tqdm(
            range(n_patch_batches),
            desc=f'Patches {sample_idx}/{len(val_loader)}',
            leave=False,
            dynamic_ncols=True
        )

        for batch_idx in inner:
            start = batch_idx * val_patch_batch
            end = min(start + val_patch_batch, patch_count)

            dem_bic = dem_bic_all[start:end]
            lidar_delta = lidar_delta_all[start:end]
            mask = mask_all[start:end]
            dist_map = dist_map_all[start:end]

            pred_dem, alpha, r_topo, r_anchor = model(
                dem_bic,
                lidar_delta,
                mask,
                dist_map
            )

            patch_preds.append(pred_dem)

            merger.add_batch(
                pred_dem.detach().cpu(),
                patch_mean_all[start:end],
                coords_all[start:end].cpu()
            )

            inner.set_postfix({
                'batch': f'{batch_idx + 1}/{n_patch_batches}',
            })

        pred_all = torch.cat(patch_preds, dim=0)
        lidar_raw_all = dem_bic_all + lidar_delta_all

        val_loss = criterion(
            pred_all,
            gt_dem_all,
            lidar_raw_all,
            mask_all,
            dist_map_all,
        )

        mask_sum_all = mask_all.sum()
        if mask_sum_all > 0:
            val_anchor_mae = (mask_all * torch.abs(pred_all - lidar_raw_all)).sum() / mask_sum_all
        else:
            val_anchor_mae = torch.tensor(0.0, device=device)
            
        total_anchor_mae += float(val_anchor_mae.item())

        merged_pred = merger.get_final_dem(as_tensor=True).unsqueeze(0).unsqueeze(0)

        mae = F.l1_loss(merged_pred, gt_canvas)
        rmse = compute_rmse(merged_pred, gt_canvas)
        psnr = compute_psnr(merged_pred, gt_canvas)

        pred_dx = safe_conv(merged_pred, sobel_x_device)
        pred_dy = safe_conv(merged_pred, sobel_y_device)
        gt_dx = safe_conv(gt_canvas, sobel_x_device)
        gt_dy = safe_conv(gt_canvas, sobel_y_device)

        pred_slope = torch.sqrt(pred_dx ** 2 + pred_dy ** 2)
        gt_slope = torch.sqrt(gt_dx ** 2 + gt_dy ** 2)
        slope_rmse = compute_rmse(pred_slope, gt_slope)

        pred_curve = safe_conv(merged_pred, laplacian_device)
        gt_curve = safe_conv(gt_canvas, laplacian_device)
        curve_rmse = compute_rmse(pred_curve, gt_curve)

        total_loss += float(val_loss['total'].item())
        total_mae += float(mae.item())
        total_rmse += float(rmse.item())
        total_psnr += float(psnr.item())
        total_slope_rmse += float(slope_rmse.item())
        total_curve_rmse += float(curve_rmse.item())
        n_samples += 1

        outer.set_postfix({
            'mae': f'{mae.item():.3f}',
            'rmse': f'{rmse.item():.3f}',
            'psnr': f'{psnr.item():.2f}',
            'slope': f'{slope_rmse.item():.3f}',
            'curve': f'{curve_rmse.item():.3f}',
        })

    return {
        'val_total': total_loss / max(n_samples, 1),
        'val_mae': total_mae / max(n_samples, 1),
        'val_rmse': total_rmse / max(n_samples, 1),
        'val_psnr': total_psnr / max(n_samples, 1),
        'val_slope_rmse': total_slope_rmse / max(n_samples, 1),
        'val_curve_rmse': total_curve_rmse / max(n_samples, 1),
        'val_anchor_mae': total_anchor_mae / max(n_samples, 1),
    }


# -----------------------------------------------------------------------------
# Main training cell
# -----------------------------------------------------------------------------

def main():
    train_loader, val_loader = create_curriculum_dataloaders(
        TRAIN_DIRS,
        VAL_DIRS,
        batch_size=BATCH_SIZE,
        num_workers=8,        
        prefetch_factor=2,    
        pin_memory=True
    )

    model = DistanceGatedGeoVDSR(topo_layers=TOPO_LAYERS, features=FEATURES).to(device)
    criterion = DistanceGatedTopoLoss(
        alpha=LOSS_ALPHA,
        beta=LOSS_BETA,
        gamma=LOSS_GAMMA,
        lambda_pin=LOSS_LAMBDA_PIN,
        pin_beta=PIN_BETA,
        decay_radius=DECAY_RADIUS,
        buffer_size=BUFFER_SIZE,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        cooldown=2,
        min_lr=1e-6,
    )

   best_metric = float('inf')
    best_epoch = 0
    stale_epochs = 0
    train_history = []
    start_epoch = 1

    # =========================================================================
    # CHECKPOINT RESUME LOGIC
    # =========================================================================
    if RESUME_CHECKPOINT is not None and Path(RESUME_CHECKPOINT).exists():
        print(f"Loading checkpoint: {RESUME_CHECKPOINT}")
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
        
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if ckpt['scheduler_state_dict']:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            
        start_epoch = ckpt['epoch'] + 1
        best_metric = ckpt.get('best_metric', float('inf'))
        best_epoch = ckpt.get('best_epoch', 0)
        train_history = ckpt.get('train_history', [])
        
        print(f"Resumed successfully. Curriculum will continue from epoch {start_epoch}.")
    # =========================================================================

    print(f'Train files: {len(train_loader.dataset)}')
    print(f'Val files:   {len(val_loader.dataset)}')

    # Update the range to use 'start_epoch' instead of 1
    epoch_bar = tqdm(range(start_epoch, EPOCHS + 1), desc='Epochs', dynamic_ncols=True)

    for epoch in epoch_bar:
        t0 = time.time()

        # ==============================================================
        # THE DENSITY-AWARE CURRICULUM SCHEDULER
        # ==============================================================
        if epoch <= 50:
            # Phase 1: Burn-in (Dense Tracks)
            train_loader.dataset.photon_focus = 0.90
            train_loader.dataset.target_photons = 50   
        elif epoch <= 150:
            # Phase 2: Propagation (Sparse Tracks)
            train_loader.dataset.photon_focus = 0.50
            train_loader.dataset.target_photons = 10   
        else:
            # Phase 3: Generalization (Empty/Natural)
            train_loader.dataset.photon_focus = 0.10
            train_loader.dataset.target_photons = 0    
        # ==============================================================

        # Pass native float32 arguments
        train_stats = train_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            train_loader=train_loader,
            device=device,
            grad_clip_norm=GRAD_CLIP_NORM,
        )

        val_stats = validate_one_epoch(
            model=model,
            criterion=criterion,
            val_loader=val_loader,
            device=device,
            val_patch_batch=VAL_PATCH_BATCH,
        )

        scheduler.step(val_stats['val_slope_rmse'])

        lr_now = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0

        row = {
            'epoch': epoch,
            'lr': lr_now,
            'train_total': train_stats['total'],
            'train_base': train_stats['base'],
            'train_slope': train_stats['slope'],
            'train_curve': train_stats['curve'],
            'train_pin': train_stats['pin'],
            'train_mae': train_stats['mae'],
            'train_rmse': train_stats['rmse'],
            'val_total': val_stats['val_total'],
            'val_mae': val_stats['val_mae'],
            'val_rmse': val_stats['val_rmse'],
            'val_psnr': val_stats['val_psnr'],
            'val_slope_rmse': val_stats['val_slope_rmse'],
            'val_curve_rmse': val_stats['val_curve_rmse'],
            'time_sec': elapsed,
        }
        train_history.append(row)

        is_best = val_stats[BEST_METRIC_NAME] < best_metric
        if is_best:
            best_metric = val_stats[BEST_METRIC_NAME]
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1

        ckpt_path = save_checkpoint(
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_metric=best_metric,
            best_epoch=best_epoch,
            train_history=train_history,
            checkpoint_dir=CHECKPOINT_DIR,
            training_config=training_config,
            is_best=is_best,
        )

        epoch_bar.set_postfix({
            'train': f"{train_stats['total']:.4f}",
            'val_mae': f"{val_stats['val_mae']:.3f}",
            'val_rmse': f"{val_stats['val_rmse']:.3f}",
            'psnr': f"{val_stats['val_psnr']:.2f}",
            'best': f"{best_metric:.3f}",
            'lr': f"{lr_now:.2e}",
        })

        tqdm.write(
            f"Epoch {epoch:03d} | "
            f"TrainLoss {train_stats['total']:.4f} | "
            f"ValAnchorMAE {val_stats['val_anchor_mae']:.4f} | "
            f"ValMAE {val_stats['val_mae']:.4f} | "
            f"SlopeRMSE {val_stats['val_slope_rmse']:.4f} | "
            f"LR {lr_now:.2e} | "
            f"Best {best_metric:.4f} @ ep {best_epoch} | "
            f"time {elapsed:.1f}s"
        )

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            tqdm.write(f"Early stopping triggered at epoch {epoch}.")
            break

    epoch_bar.close()
    print('Training finished.')

if __name__ == '__main__':
    main()